In [ ]:
# ============================================================
# 30. ROBUST 3D ENM VISUALIZATION
# ============================================================

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from IPython.display import display, HTML
from matplotlib.animation import FuncAnimation
from IPython.display import HTML as IPyHTML

# ------------------------------------------------------------
# COLAB RENDERER
# ------------------------------------------------------------

try:
    pio.renderers.default = "colab"
except Exception:
    pass


# ============================================================
# 31. 3D EDGE BUILDER
# ============================================================

def build_3d_edge_trace(
    coords,
    graph,
    line_width=2
):
    xs = []
    ys = []
    zs = []

    for i, j in graph.edges():

        xs += [
            float(coords[i, 0]),
            float(coords[j, 0]),
            None
        ]

        ys += [
            float(coords[i, 1]),
            float(coords[j, 1]),
            None
        ]

        zs += [
            float(coords[i, 2]),
            float(coords[j, 2]),
            None
        ]

    return go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="lines",
        line=dict(
            width=line_width
        ),
        hoverinfo="skip",
        name="ENM contacts"
    )


# ============================================================
# 32. 3D BACKBONE
# ============================================================

def build_3d_backbone_trace(
    coords
):
    return go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="lines",
        line=dict(
            width=5
        ),
        hoverinfo="skip",
        name="Cα backbone"
    )


# ============================================================
# 33. 3D C-alpha NODES
# ============================================================

def build_3d_node_trace(
    coords
):
    return go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers",
        marker=dict(
            size=5
        ),
        text=[
            f"Residue {i + 1}"
            for i in range(len(coords))
        ],
        hovertemplate=(
            "%{text}"
            "<br>X = %{x:.2f} Å"
            "<br>Y = %{y:.2f} Å"
            "<br>Z = %{z:.2f} Å"
            "<extra></extra>"
        ),
        name="Cα"
    )


# ============================================================
# 34. STATIC NATIVE 3D
# ============================================================

def render_native_3d():
    fig = go.Figure()

    fig.add_trace(
        build_3d_edge_trace(
            coords_target,
            G_target_native,
            line_width=2
        )
    )

    fig.add_trace(
        build_3d_backbone_trace(
            coords_target
        )
    )

    fig.add_trace(
        build_3d_node_trace(
            coords_target
        )
    )

    fig.update_layout(
        title=(
            "1U3C — Native 3D Elastic Network"
            f"<br>Cutoff = {CUTOFF_A:.1f} Å"
        ),
        width=1100,
        height=850,
        scene=dict(
            xaxis_title="X (Å)",
            yaxis_title="Y (Å)",
            zaxis_title="Z (Å)",
            aspectmode="data"
        )
    )

    # Self-contained HTML prevents renderer failure.
    html = fig.to_html(
        full_html=False,
        include_plotlyjs=True
    )

    display(
        HTML(html)
    )


render_native_3d()


# ============================================================
# 35. STATIC RANDOMIZED 3D
# ============================================================

def render_randomized_3d():
    fig = go.Figure()

    fig.add_trace(
        build_3d_edge_trace(
            coords_target,
            G_target_random,
            line_width=2
        )
    )

    fig.add_trace(
        build_3d_backbone_trace(
            coords_target
        )
    )

    fig.add_trace(
        build_3d_node_trace(
            coords_target
        )
    )

    fig.update_layout(
        title=(
            "1U3C — Degree-Preserving "
            "Maslov–Sneppen 3D Network"
        ),
        width=1100,
        height=850,
        scene=dict(
            xaxis_title="X (Å)",
            yaxis_title="Y (Å)",
            zaxis_title="Z (Å)",
            aspectmode="data"
        )
    )

    html = fig.to_html(
        full_html=False,
        include_plotlyjs=True
    )

    display(
        HTML(html)
    )


render_randomized_3d()


# ============================================================
# 36. SELECT ENM EIGENMODE
# ============================================================

MODE_INDEX_3D = DISPLAY_MODE_RANK - 1

if MODE_INDEX_3D < 0:
    raise ValueError(
        "DISPLAY_MODE_RANK must be >= 1."
    )

if MODE_INDEX_3D >= len(
    omega_target_native
):
    raise ValueError(
        "Selected eigenmode does not exist."
    )


omega_3d = float(
    omega_target_native[
        MODE_INDEX_3D
    ]
)


# ------------------------------------------------------------
# EXTRACT EIGENVECTOR
# ------------------------------------------------------------

raw_mode = np.real(
    vec_target_native[
        :,
        MODE_INDEX_3D
    ]
)


raw_mode = raw_mode.reshape(
    (-1, 3)
)


# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

node_norms = np.linalg.norm(
    raw_mode,
    axis=1
)

mode_norm = float(
    np.max(node_norms)
)

if not np.isfinite(mode_norm):
    raise ValueError(
        "Eigenmode contains non-finite values."
    )

if mode_norm <= 0.0:
    raise ValueError(
        "Selected eigenmode has zero amplitude."
    )

mode_3d = raw_mode / mode_norm


# ============================================================
# 37. DEFINE INITIAL 3D COORDINATES
# ============================================================
#
# THIS IS THE VARIABLE THAT WAS MISSING.
#
# ============================================================

phase_initial = 0.0

initial_displacement_3d = (
    EIGENMODE_AMPLITUDE_A
    * np.sin(phase_initial)
    * mode_3d
)

coords_initial_3d = (
    coords_target
    + initial_displacement_3d
)


# ============================================================
# 38. EIGENMODE FRAME GENERATOR
# ============================================================

def generate_eigenmode_frames(
    base_coords,
    mode,
    graph,
    amplitude,
    n_frames
):

    phases = np.linspace(
        0.0,
        2.0 * np.pi,
        n_frames,
        endpoint=False
    )

    frames = []

    for frame_id, phase in enumerate(phases):

        displacement = (
            amplitude
            * np.sin(phase)
            * mode
        )

        displaced_coords = (
            base_coords
            + displacement
        )

        edge_trace = (
            build_3d_edge_trace(
                displaced_coords,
                graph,
                line_width=2
            )
        )

        backbone_trace = (
            build_3d_backbone_trace(
                displaced_coords
            )
        )

        node_trace = (
            build_3d_node_trace(
                displaced_coords
            )
        )

        frames.append(
            go.Frame(
                data=[
                    edge_trace,
                    backbone_trace,
                    node_trace
                ],
                name=f"frame_{frame_id}"
            )
        )

    return phases, frames


phases_3d, frames_3d = (
    generate_eigenmode_frames(
        coords_target,
        mode_3d,
        G_target_native,
        EIGENMODE_AMPLITUDE_A,
        N_ANIMATION_FRAMES
    )
)


# ============================================================
# 39. CREATE ANIMATED 3D FIGURE
# ============================================================

fig_3d = go.Figure(
    data=[
        build_3d_edge_trace(
            coords_initial_3d,
            G_target_native,
            line_width=2
        ),

        build_3d_backbone_trace(
            coords_initial_3d
        ),

        build_3d_node_trace(
            coords_initial_3d
        )
    ],

    frames=frames_3d
)


# ============================================================
# 40. ANIMATION BUTTONS
# ============================================================

fig_3d.update_layout(
    title=(
        "1U3C — 3D ENM Eigenmode Simulation"
        f"<br>Mode #{DISPLAY_MODE_RANK}"
        f" | ω = {omega_3d:.6f}"
        f" | A = {EIGENMODE_AMPLITUDE_A:.2f} Å"
    ),

    width=1100,
    height=850,

    scene=dict(
        xaxis_title="X (Å)",
        yaxis_title="Y (Å)",
        zaxis_title="Z (Å)",
        aspectmode="data"
    ),

    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.02,
            y=1.08,
            showactive=False,

            buttons=[

                # PLAY
                dict(
                    label="▶ PLAY",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=60,
                                redraw=True
                            ),
                            transition=dict(
                                duration=0
                            ),
                            fromcurrent=True
                        )
                    ]
                ),

                # PAUSE
                dict(
                    label="⏸ PAUSE",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            mode="immediate",
                            frame=dict(
                                duration=0,
                                redraw=False
                            )
                        )
                    ]
                )
            ]
        )
    ]
)


# ============================================================
# 41. SLIDER
# ============================================================

slider_steps = []

for frame_id in range(
    N_ANIMATION_FRAMES
):

    slider_steps.append(
        dict(
            method="animate",
            label=str(frame_id + 1),
            args=[
                [f"frame_{frame_id}"],
                dict(
                    mode="immediate",
                    frame=dict(
                        duration=0,
                        redraw=True
                    ),
                    transition=dict(
                        duration=0
                    )
                )
            ]
        )
    )


fig_3d.update_layout(
    sliders=[
        dict(
            active=0,
            x=0.08,
            y=0.01,
            len=0.82,

            currentvalue=dict(
                prefix="Frame: "
            ),

            steps=slider_steps
        )
    ]
)


# ============================================================
# 42. FORCE INLINE 3D RENDER
# ============================================================

html_3d = fig_3d.to_html(
    full_html=False,
    include_plotlyjs=True
)

display(
    HTML(html_3d)
)


# ============================================================
# 43. MATPLOTLIB 3D FALLBACK
# ============================================================

def render_matplotlib_3d_fallback():

    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation

    fig = plt.figure(
        figsize=(11, 9)
    )

    ax = fig.add_subplot(
        111,
        projection="3d"
    )

    # Initial position is explicitly defined.
    coords0 = coords_initial_3d.copy()

    # Nodes
    scatter = ax.scatter(
        coords0[:, 0],
        coords0[:, 1],
        coords0[:, 2],
        s=18
    )

    # Backbone
    backbone_line, = ax.plot(
        coords0[:, 0],
        coords0[:, 1],
        coords0[:, 2],
        linewidth=1.5
    )

    # Contact edges
    edge_lines = []

    for i, j in G_target_native.edges():

        line, = ax.plot(
            [
                coords0[i, 0],
                coords0[j, 0]
            ],
            [
                coords0[i, 1],
                coords0[j, 1]
            ],
            [
                coords0[i, 2],
                coords0[j, 2]
            ],
            linewidth=0.6
        )

        edge_lines.append(
            (line, i, j)
        )

    # --------------------------------------------------------
    # Keep axis limits fixed throughout animation.
    # --------------------------------------------------------

    margin = (
        EIGENMODE_AMPLITUDE_A
        + 2.0
    )

    xmin = float(
        np.min(coords_target[:, 0])
        - margin
    )

    xmax = float(
        np.max(coords_target[:, 0])
        + margin
    )

    ymin = float(
        np.min(coords_target[:, 1])
        - margin
    )

    ymax = float(
        np.max(coords_target[:, 1])
        + margin
    )

    zmin = float(
        np.min(coords_target[:, 2])
        - margin
    )

    zmax = float(
        np.max(coords_target[:, 2])
        + margin
    )

    ax.set_xlim(
        xmin,
        xmax
    )

    ax.set_ylim(
        ymin,
        ymax
    )

    ax.set_zlim(
        zmin,
        zmax
    )

    ax.set_xlabel(
        "X (Å)"
    )

    ax.set_ylabel(
        "Y (Å)"
    )

    ax.set_zlabel(
        "Z (Å)"
    )

    ax.set_title(
        "1U3C — 3D ENM Eigenmode "
        f"#{DISPLAY_MODE_RANK}"
    )

    # --------------------------------------------------------
    # Animation
    # --------------------------------------------------------

    phases = np.linspace(
        0.0,
        2.0 * np.pi,
        N_ANIMATION_FRAMES,
        endpoint=False
    )

    def update(frame_id):

        phase = phases[
            frame_id
        ]

        displacement = (
            EIGENMODE_AMPLITUDE_A
            * np.sin(phase)
            * mode_3d
        )

        coords = (
            coords_target
            + displacement
        )

        # Nodes
        scatter._offsets3d = (
            coords[:, 0],
            coords[:, 1],
            coords[:, 2]
        )

        # Backbone
        backbone_line.set_data(
            coords[:, 0],
            coords[:, 1]
        )

        backbone_line.set_3d_properties(
            coords[:, 2]
        )

        # ENM edges
        for line, i, j in edge_lines:

            line.set_data(
                [
                    coords[i, 0],
                    coords[j, 0]
                ],
                [
                    coords[i, 1],
                    coords[j, 1]
                ]
            )

            line.set_3d_properties(
                [
                    coords[i, 2],
                    coords[j, 2]
                ]
            )

        return (
            [scatter, backbone_line]
            +
            [
                item[0]
                for item in edge_lines
            ]
        )

    animation = FuncAnimation(
        fig,
        update,
        frames=N_ANIMATION_FRAMES,
        interval=60,
        blit=False
    )

    plt.close(fig)

    return animation


# ============================================================
# 44. FALLBACK DISPLAY
# ============================================================

# The Plotly version above is the primary visualization.
# Matplotlib fallback is generated independently and therefore
# does not depend on coords_initial_3d being created elsewhere.

try:

    fallback_animation = (
        render_matplotlib_3d_fallback()
    )

    display(
        IPyHTML(
            fallback_animation.to_jshtml()
        )
    )

except Exception as fallback_error:

    print(
        "3D fallback error:",
        repr(fallback_error)
    )


# ============================================================
# 45. FINAL 3D REPORT
# ============================================================

print("\n" + "=" * 72)
print("3D ENM SIMULATION — READY")
print("=" * 72)

print(
    f"Structure              : {TARGET_PDB}"
)

print(
    f"Cα nodes               : {len(coords_target)}"
)

print(
    f"ENM contacts            : "
    f"{G_target_native.number_of_edges()}"
)

print(
    f"Cutoff                  : "
    f"{CUTOFF_A:.2f} Å"
)

print(
    f"Mode                    : "
    f"#{DISPLAY_MODE_RANK}"
)

print(
    f"ω                       : "
    f"{omega_3d:.8f}"
)

print(
    f"Visual amplitude        : "
    f"{EIGENMODE_AMPLITUDE_A:.3f} Å"
)

print(
    f"Animation frames        : "
    f"{N_ANIMATION_FRAMES}"
)

print(
    "3D Plotly               : READY"
)

print(
    "3D Matplotlib fallback  : READY"
)

print("=" * 72)